<a href="https://colab.research.google.com/github/solasobambo-prog/ts-data-science/blob/main/Module3_kaggle.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Week 3a → 3d Capstone Checklist: NYC Airbnb Open Data

1. **Week 3a (Deduplication, Whitespace & Types):** Run primary-key duplicate check on `id` (`listing_id`), strip whitespace on `neighbourhood_group` and `room_type` while logging before/after unique counts, coerce `price` to numeric float, and parse `last_review` to `datetime` (deliberately keeping unreviewed dates as `NaT`).
2. **Week 3b (Missingness & Structural Missingness Test):** Audit missing values across all 16 columns from `raw_copy`, prove structural missingness by testing whether `last_review` and `reviews_per_month` miss on identical `number_of_reviews == 0` rows, and flag `reviews_missing` before filling.
3. **Week 3c (Global & Stratified Outlier Detection):** Compute the global upper IQR fence dynamically from quantiles (closing the prediction check), then calculate **stratified IQR fences by `room_type`** so each tier is judged against its own peers.
4. **Week 3c/3d (Capping & Anomaly Flags):** Flag `$0` listings (`price == 0`) and stratified upper-fence outliers without deleting rows, and create `price_capped_strat` by clipping each `room_type` at its own upper fence.
5. **Week 3d (Before/After Snapshot & Honest Reporting):** Generate a unified `before_after` summary table comparing raw vs. cleaned metrics, write a strict three-sentence summary naming the statistic that moved most, and place deeper exploratory findings in a dedicated section.


In [31]:
import os
import kagglehub
import pandas as pd
import numpy as np

# 1. Download & Load
path = kagglehub.dataset_download("dgomonov/new-york-city-airbnb-open-data")
df = pd.read_csv(os.path.join(path, "AB_NYC_2019.csv"))

# Capture untouched raw snapshot for restart-and-run-all discipline
raw_copy = df.copy()

# --- PREDICTIONS BEFORE RUNNING ---
# Prediction 1 (Duplicates): 0 exact duplicates and 0 duplicate listing 'id's, since 'id' is a unique database primary key.
# Prediction 2 (Missingness): 'last_review' and 'reviews_per_month' will each have ~10,000 missing rows (~20%) on identical zero-review listings; 'name' and 'host_name' will have <50 missing; all other columns 0.
# Prediction 3 (Global Price IQR Outliers): With n=48,895 and heavy right skew, global IQR fences will flag roughly 2,500–3,500 high-price outliers (~5%–7%).

print("=== WEEK 3a: DUPLICATES, WHITESPACE & TYPE COERCION ===")
exact_dups = df.duplicated().sum()
id_dups = df.duplicated(subset=['id']).sum()
print(f"Exact duplicate rows: {exact_dups} | Duplicate listing IDs: {id_dups}")
# Prediction 1 check: Held — 0 exact duplicates and 0 duplicate listing IDs found.

# Transformation 1: Coerce price to float and strip whitespace on categorical columns
# Justification: Numeric coercion guarantees arithmetic safety, while stripping whitespace prevents silent category fragmentation.
pre_coerce_na = df['price'].isna().sum()
df['price'] = pd.to_numeric(df['price'], errors='coerce')
new_coerce_na = df['price'].isna().sum() - pre_coerce_na

ng_before, ng_after = df['neighbourhood_group'].nunique(), df['neighbourhood_group'].str.strip().nunique()
rt_before, rt_after = df['room_type'].nunique(), df['room_type'].str.strip().nunique()
df['neighbourhood_group'] = df['neighbourhood_group'].str.strip()
df['room_type'] = df['room_type'].str.strip()

print(f"Price coercion new NaNs: {new_coerce_na} (checked and clean)")
print(f"Unique counts pre/post strip -> neighbourhood_group: {ng_before}->{ng_after} | room_type: {rt_before}->{rt_after} (checked and clean)")

# Transformation 2: Parse last_review to datetime and extract year
# Justification: Converting text dates to datetime enables temporal analysis; unreviewed rows are deliberately left as NaT (and NaN in last_review_year) because inventing a date for a listing with zero reviews would fabricate history.
df['last_review'] = pd.to_datetime(df['last_review'], errors='coerce')
df['last_review_year'] = df['last_review'].dt.year

print("\n=== WEEK 3b: MISSINGNESS & STRUCTURAL TEST ===")
missing_report = pd.DataFrame({
    'missing_count': raw_copy.isna().sum(),
    'missing_pct': (raw_copy.isna().mean() * 100).round(2)
}).sort_values(by='missing_count', ascending=False)
display(missing_report.head(6))
# Prediction 2 check: Held — last_review and reviews_per_month each have 10,052 missing (20.56%), host_name has 21, name has 16, and all other columns have 0.

# Test structural missingness directly on raw_copy
same_rows = raw_copy['last_review'].isna().equals(raw_copy['reviews_per_month'].isna())
zero_rows = raw_copy['reviews_per_month'].isna().equals(raw_copy['number_of_reviews'] == 0)
print(f"Structural missingness check -> same_rows: {same_rows} | zero_rows: {zero_rows}")

# Transformation 3: Flag reviews_missing BEFORE filling
# Justification: Preserves unreviewed status as an informative behavioral feature before imputing 0.
df['reviews_missing'] = df['reviews_per_month'].isna()

# Informative missingness comparison (not a causal NMAR proof, but shows unreviewed listings differ in price and availability)
informative_check = df.groupby('reviews_missing').agg(
    n=('id', 'count'),
    median_price=('price', 'median'),
    mean_price=('price', 'mean'),
    median_availability=('availability_365', 'median')
).round(2)
print("\nInformative Missingness Comparison (reviews_missing = True vs False):")
display(informative_check)

# Transformation 4: Fill reviews_per_month with 0 and missing text names with 'Unknown'
# Justification: Verified structurally missing zero-review listings have 0 reviews per month, and 'Unknown' retains all 37 name-missing rows without dropping valid pricing data.
df['reviews_per_month'] = df['reviews_per_month'].fillna(0)
df['host_name'] = df['host_name'].fillna('Unknown')
df['name'] = df['name'].fillna('Unknown')


Using Colab cache for faster access to the 'new-york-city-airbnb-open-data' dataset.
=== WEEK 3a: DUPLICATES, WHITESPACE & TYPE COERCION ===
Exact duplicate rows: 0 | Duplicate listing IDs: 0
Price coercion new NaNs: 0 (checked and clean)
Unique counts pre/post strip -> neighbourhood_group: 5->5 | room_type: 3->3 (checked and clean)

=== WEEK 3b: MISSINGNESS & STRUCTURAL TEST ===


,missing_count,missing_pct
last_review,10052,20.56
reviews_per_month,10052,20.56
host_name,21,0.04
name,16,0.03
neighbourhood_group,0,0.00
neighbourhood,0,0.00


Structural missingness check -> same_rows: True | zero_rows: True

Informative Missingness Comparison (reviews_missing = True vs False):


,n,median_price,mean_price,median_availability
reviews_missing,,,,
False,38843,101.0,142.32,55.0
True,10052,120.0,192.92,6.0


In [32]:
print("=== WEEK 3c: GLOBAL & STRATIFIED IQR FENCES ===")
# Compute global fences dynamically from quantiles
g_q1 = df['price'].quantile(0.25)
g_q3 = df['price'].quantile(0.75)
g_iqr = g_q3 - g_q1
g_upper = g_q3 + 1.5 * g_iqr
global_outliers_count = (df['price'] > g_upper).sum()

print(f"Global Upper Fence (computed): ${g_upper:.2f} | Outliers flagged: {global_outliers_count} ({global_outliers_count/len(df)*100:.2f}%)")
# Prediction 3 check: Held — 2,972 outliers (6.08%) fall inside our predicted 2,500–3,500 (5%–7%) range.

# Calculate stratified IQR fences by room_type
strat_fences = df.groupby('room_type')['price'].agg(
    n='count',
    q1=lambda s: s.quantile(0.25),
    median='median',
    q3=lambda s: s.quantile(0.75)
)
strat_fences['iqr'] = strat_fences['q3'] - strat_fences['q1']
strat_fences['upper_fence'] = strat_fences['q3'] + 1.5 * strat_fences['iqr']
display(strat_fences.round(2))

df['room_upper_fence'] = df['room_type'].map(strat_fences['upper_fence'])

# Transformation 5: Flag stratified outliers and $0 free listings without deleting rows
# Justification: Stratified flags judge each listing against its room-type peers, while flagging price == 0 isolates potential pricing errors for data-owner review.
df['is_strat_outlier'] = df['price'] > df['room_upper_fence']
df['is_zero_price'] = df['price'] == 0

print(f"Stratified IQR outliers (by room_type): {df['is_strat_outlier'].sum()}")
print(f"Zero-price listings flagged (price == 0): {df['is_zero_price'].sum()}")

# Transformation 6: Create stratified capped price column for modeling
# Justification: Clipping each listing at its own room_type upper fence neutralizes extreme tail leverage while keeping raw 'price' intact for financial reporting.
df['price_capped_strat'] = np.minimum(df['price'], df['room_upper_fence'])

print("\n=== WEEK 3d: BEFORE AND AFTER SNAPSHOT TABLE ===")
def snap(d, price_col):
    s = d[price_col]
    return pd.Series({
        'rows': len(d),
        'missing_cells': int(d.isna().sum().sum()),
        'price_mean': s.mean(),
        'price_median': s.median(),
        'price_iqr': s.quantile(0.75) - s.quantile(0.25),
        'price_std': s.std(),
        'price_max': s.max(),
    })

before_after = pd.DataFrame({
    'before': snap(raw_copy, 'price'),
    'after': snap(df, 'price_capped_strat'),
})
display(before_after.round(2))


=== WEEK 3c: GLOBAL & STRATIFIED IQR FENCES ===
Global Upper Fence (computed): $334.00 | Outliers flagged: 2972 (6.08%)


,n,q1,median,q3,iqr,upper_fence
room_type,,,,,,
Entire home/apt,25409,120.0,160.0,229.0,109.0,392.5
Private room,22326,50.0,70.0,95.0,45.0,162.5
Shared room,1160,33.0,45.0,75.0,42.0,138.0


Stratified IQR outliers (by room_type): 3244
Zero-price listings flagged (price == 0): 11

=== WEEK 3d: BEFORE AND AFTER SNAPSHOT TABLE ===


,before,after
rows,48895.00,48895.00
missing_cells,20141.00,20104.00
price_mean,152.72,133.24
price_median,106.00,106.00
price_iqr,106.00,96.00
price_std,240.15,87.90
price_max,10000.00,392.50


### Three-Sentence Before & After Summary

1. **Rows In:** The pipeline ingested **48,895 rows** across 16 columns from `AB_NYC_2019.csv` (`raw_copy`), with `0` duplicate IDs and **20,141 total missing cells**.
2. **Rows Out:** All **48,895 rows** were retained (`0` deleted) after flagging unreviewed listings (`reviews_missing`), imputing verified structural zeros/placeholders, and deliberately keeping `last_review` and `last_review_year` as `NaT`/`NaN` for unreviewed rows.
3. **Statistic That Moved Most:** While `price_max` fell **96.1%** from **$10,000.00** to **$392.50** by construction (the largest room-type fence), the more informative move was `price_std` dropping **63.4%** from **$240.15** to **$87.90** while `price_median` held steady at **$106.00**.


In [33]:
print("=== EXPLORATORY SECTION: MINIMUM NIGHTS, INVENTORY MIX & WITHIN-TIER PRICING ===")

# 1. Minimum nights & raw price skewness
print(f"Raw Price Skewness: {df['price'].skew():.2f}")
print(f"Minimum Nights -> Median: {df['minimum_nights'].median():.0f} | Q3: {df['minimum_nights'].quantile(0.75):.0f} | Max: {df['minimum_nights'].max()} | Mean: {df['minimum_nights'].mean():.2f}")

# 2. Room-type inventory mix by borough (%)
print("\n--- Room Type Mix by Borough (%) ---")
room_mix = (pd.crosstab(df['neighbourhood_group'], df['room_type'], normalize='index') * 100).round(1)
display(room_mix)

# 3. Within-tier test: Does Manhattan cost more even within the exact same room_type?
print("\n--- Honest Median & IQR Price by Borough and Room Type ---")
honest_report = df.groupby(['neighbourhood_group', 'room_type'])['price'].agg(
    n='count',
    median='median',
    iqr=lambda s: s.quantile(0.75) - s.quantile(0.25)
).round(2)
display(honest_report)


=== EXPLORATORY SECTION: MINIMUM NIGHTS, INVENTORY MIX & WITHIN-TIER PRICING ===
Raw Price Skewness: 19.12
Minimum Nights -> Median: 3 | Q3: 5 | Max: 1250 | Mean: 7.03

--- Room Type Mix by Borough (%) ---


room_type,Entire home/apt,Private room,Shared room
neighbourhood_group,,,
Bronx,34.7,59.8,5.5
Brooklyn,47.5,50.4,2.1
Manhattan,60.9,36.8,2.2
Queens,37.0,59.5,3.5
Staten Island,47.2,50.4,2.4



--- Honest Median & IQR Price by Borough and Room Type ---


n  median     iqr
neighbourhood_group room_type                             
Bronx               Entire home/apt    379   100.0   60.00
                    Private room       652    53.5   30.00
                    Shared room         60    40.0   27.50
Brooklyn            Entire home/apt   9559   145.0   94.00
                    Private room     10132    65.0   30.00
                    Shared room        413    36.0   20.00
Manhattan           Entire home/apt  13199   191.0  116.50
                    Private room      7982    90.0   52.00
                    Shared room        480    69.0   40.25
Queens              Entire home/apt   2096   120.0   78.25
                    Private room      3372    60.0   28.00
                    Shared room        198    37.0   24.25
Staten Island       Entire home/apt    176   100.0   76.25
                    Private room       188    50.0   35.00
                    Shared room          9    30.0   46.00

### Exploratory Notes: Inventory Mix vs. Location Effect

* **Minimum Nights Skew:** While the median minimum stay is `3` nights (`Q3 = 5`), extreme listings reach `1,250` nights, pulling the mean to `7.03` nights.
* **Inventory Mix & Within-Tier Pricing:** Although **60.9%** of Manhattan listings are `Entire home/apt` (compared to `34.7%–47.5%` in other boroughs), the two-way `neighbourhood_group × room_type` table shows that inventory mix is **not** the sole factor: Manhattan also has a higher median price *within every individual room type* (`$191.00` for Manhattan entire homes vs. `$145.00` in Brooklyn and `$120.00` in Queens).
